# Module 2: Exploratory Analysis & Distribution Fitting

**QuantVerse** — Quantitative Portfolio Intelligence System

---

## Objectives

In this notebook we:

1. Test normality assumptions on asset return distributions
2. Fit alternative distributions (Student-t, skewed-t, etc.) and compare via AIC
3. Visualize deviations from normality using QQ-plots
4. Analyze higher moments (skewness, kurtosis) across asset classes
5. Compute rolling statistics to detect regime changes
6. Model volatility clustering with GARCH(1,1)
7. Verify stylized facts of financial returns
8. Analyze correlation regimes (crisis vs calm)
9. Perform PCA to identify dominant risk factors
10. Measure correlation stability over time

### Why This Matters for Portfolio Construction

Most portfolio optimization (Markowitz) assumes returns are normally distributed.
If returns are NOT normal — and they almost never are — we need:
- **Robust covariance estimators** (Module 3)
- **Tail-aware risk measures** like CVaR instead of VaR
- **Regime-aware allocation** that adapts to changing correlations

---

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import logging
import sys
import os
import json

sys.path.insert(0, os.path.abspath('..'))

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_palette('husl')

print('Setup complete.')

In [ ]:
# Load processed data from Module 1
data_dir = '../data/processed'

daily_returns = pd.read_parquet(f'{data_dir}/returns_daily.parquet')
log_returns = pd.read_parquet(f'{data_dir}/log_returns_daily.parquet')
clean_prices = pd.read_parquet(f'{data_dir}/prices_clean.parquet')

with open(f'{data_dir}/asset_class_map.json', 'r') as f:
    class_map = json.load(f)

# Filter out signal tickers for analysis
signal_tickers = [t for t, c in class_map.items() if c == 'signals']
investable = [t for t in daily_returns.columns if t not in signal_tickers]

print(f'Loaded {len(daily_returns.columns)} assets, {len(daily_returns)} trading days')
print(f'Date range: {daily_returns.index[0].date()} to {daily_returns.index[-1].date()}')
print(f'Investable assets: {len(investable)}')

## 1. Normality Tests

We apply four statistical tests of normality:
- **Jarque-Bera**: tests skewness = 0 and kurtosis = 3 jointly
- **Shapiro-Wilk**: most powerful test for smaller samples
- **Anderson-Darling**: sensitive to tail deviations
- **D'Agostino-Pearson**: omnibus test combining skewness and kurtosis

If returns were truly normal, we'd expect some assets to pass at $\alpha = 0.05$.
Spoiler: almost none will.

In [ ]:
from project.exploratory import ReturnAnalyzer

analyzer = ReturnAnalyzer(daily_returns, asset_class_map=class_map)

normality = analyzer.normality_tests(alpha=0.05)
print('Normality Test Results (alpha=0.05):')
print('=' * 80)
print(normality[['JB_pvalue', 'SW_pvalue', 'AD_stat', 'K2_pvalue', 'Tests_Passed', 'Total_Tests']].to_string())
print(f'\nAssets passing ALL normality tests: {(normality["Tests_Passed"] == normality["Total_Tests"]).sum()} / {len(normality)}')

## 2. Distribution Fitting

Since returns are NOT normal, which distribution fits best?
We compare: Normal, Student-t, Skew-Normal, Generalized Normal, and Laplace.

We select the best fit using **AIC** (Akaike Information Criterion) — lower is better.

In [ ]:
fit_results = analyzer.fit_distributions()
best_fits = analyzer.best_fit_summary(fit_results)

print('Best-Fitting Distribution per Asset (by AIC):')
print('=' * 70)
print(best_fits.to_string())
print(f'\nDistribution frequency:')
print(best_fits['Distribution'].value_counts().to_string())

## 3. QQ-Plots

Visual check: if returns were normal, QQ-plots would be perfectly linear.
Deviations at the tails indicate fat tails (heavy-tailed distributions).

In [ ]:
from project.exploratory import ExploratoryVisualizer

viz = ExploratoryVisualizer(asset_class_map=class_map)

# Representative assets from each class
qq_tickers = ['XLK', 'XLF', 'EEM', 'BTC-USD', 'ETH-USD', 'SOL-USD',
              'GLD', 'USO', 'TLT', 'HYG', 'VNQ', 'SLV']
qq_tickers = [t for t in qq_tickers if t in daily_returns.columns]

fig = viz.plot_qq_grid(daily_returns, tickers=qq_tickers)
plt.show()

## 4. Return Distributions with Fitted Curves

Overlay Normal and Student-t fits on return histograms.
Student-t consistently captures the peaked center and fat tails better.

In [ ]:
fig = viz.plot_return_distributions(daily_returns, tickers=qq_tickers)
plt.show()

## 5. Higher Moments: Skewness & Kurtosis

**Skewness < 0**: left tail is heavier → larger downside risk
**Excess Kurtosis > 0**: fatter tails than normal → more extreme events

For portfolio optimization, these deviations mean:
- Gaussian VaR underestimates tail risk
- We need CVaR (Expected Shortfall) as the primary risk measure

In [ ]:
moments = analyzer.higher_moments()
print('Higher Moments Analysis:')
print('=' * 90)
print(moments[['Skewness', 'Excess_Kurtosis', 'Tail_Ratio', 'Left_Tail_5pct', 'Right_Tail_95pct', 'Asset_Class']].sort_values('Excess_Kurtosis', ascending=False).to_string())

In [ ]:
fig = viz.plot_skewness_kurtosis(moments)
plt.show()

## 6. Rolling Statistics

Returns are NOT stationary — volatility and correlations change over time.
Rolling windows reveal regime shifts that static analysis misses.

In [ ]:
rolling = analyzer.rolling_statistics(window=63)  # 3-month window

# Plot rolling volatility for representative assets
vol_tickers = ['XLK', 'BTC-USD', 'GLD', 'TLT', 'EEM', 'USO', 'VNQ', 'HYG']
vol_tickers = [t for t in vol_tickers if t in rolling['volatility'].columns]
fig = viz.plot_rolling_volatility(rolling, tickers=vol_tickers)
plt.show()

In [ ]:
# Rolling Sharpe Ratio
fig, ax = plt.subplots(figsize=(16, 6))
for ticker in ['XLK', 'BTC-USD', 'GLD', 'TLT']:
    if ticker in rolling['sharpe'].columns:
        ax.plot(rolling['sharpe'][ticker], label=ticker, alpha=0.8, lw=1.2)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Rolling 3-Month Sharpe Ratio', fontsize=14, fontweight='bold')
ax.set_ylabel('Sharpe Ratio')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 7. Volatility Clustering — GARCH(1,1)

Financial returns exhibit **volatility clustering**: large moves follow large moves.
GARCH(1,1) models this as: $\sigma_t^2 = \omega + \alpha r_{t-1}^2 + \beta \sigma_{t-1}^2$

**Persistence** = $\alpha + \beta$. Values close to 1 = long-lasting volatility shocks.

In [ ]:
# GARCH summary across investable assets
garch_tickers = [t for t in investable if t in daily_returns.columns]
garch_summary = analyzer.garch_summary(tickers=garch_tickers)

print('GARCH(1,1) Summary (sorted by Persistence):')
print('=' * 80)
print(garch_summary.sort_values('Persistence', ascending=False)[['Alpha', 'Beta', 'Persistence', 'AIC', 'Asset_Class']].to_string())

In [ ]:
# GARCH conditional volatility for BTC
btc_garch = analyzer.fit_garch('BTC-USD')
fig = viz.plot_garch_volatility(
    daily_returns['BTC-USD'].dropna(),
    btc_garch['cond_vol'],
    'BTC-USD'
)
plt.show()
print(f"BTC-USD GARCH Persistence: {btc_garch['persistence']:.4f}")

## 8. Stylized Facts of Financial Returns

We test five well-known empirical regularities:
1. **Fat tails**: excess kurtosis > 0
2. **Negative skewness**: for equities especially
3. **Volatility clustering**: autocorrelation in squared returns (Ljung-Box test)
4. **Leverage effect**: negative correlation between returns and future volatility
5. **Slow decay of absolute return autocorrelation**: long memory in volatility

In [ ]:
facts = analyzer.stylized_facts()
print('Stylized Facts Summary:')
print('=' * 100)
print(facts[['Excess_Kurtosis', 'Fat_Tails', 'Skewness', 'Negative_Skew', 
             'Vol_Clustering', 'Leverage_Corr', 'Leverage_Effect', 'AbsReturn_ACF1']].to_string())

print(f"\nFat tails: {facts['Fat_Tails'].sum()}/{len(facts)} assets")
print(f"Negative skew: {facts['Negative_Skew'].sum()}/{len(facts)} assets")
print(f"Vol clustering: {facts['Vol_Clustering'].sum()}/{len(facts)} assets")
print(f"Leverage effect: {facts['Leverage_Effect'].sum()}/{len(facts)} assets")

In [ ]:
fig = viz.plot_stylized_facts_heatmap(facts)
plt.show()

## 9. Correlation Regimes: Crisis vs Calm

**The most dangerous assumption in portfolio optimization**: that correlations are stable.

During market crises, correlations tend to increase sharply — the "diversification
disappears when you need it most" problem. We test this by comparing correlation
matrices from the worst 10% of market days vs the remaining 90%.

In [ ]:
from project.exploratory import CorrelationAnalyzer

corr_analyzer = CorrelationAnalyzer(daily_returns[investable], asset_class_map=class_map)

crisis_result = corr_analyzer.crisis_vs_calm_correlation(market_proxy='XLK', threshold_pct=10.0)

# Average correlation comparison
crisis_avg = crisis_result['crisis'].values[np.triu_indices_from(crisis_result['crisis'].values, k=1)].mean()
calm_avg = crisis_result['calm'].values[np.triu_indices_from(crisis_result['calm'].values, k=1)].mean()
print(f'Average correlation — Crisis: {crisis_avg:.3f}, Calm: {calm_avg:.3f}')
print(f'Correlation increase during crisis: {(crisis_avg/calm_avg - 1)*100:.1f}%')

In [ ]:
fig = viz.plot_crisis_vs_calm(crisis_result)
plt.show()

In [ ]:
# Average market-wide correlation over time
avg_corr_ts = corr_analyzer.average_rolling_correlation(window=63)
fig = viz.plot_avg_correlation_timeseries(avg_corr_ts)
plt.show()

## 10. PCA — Dominant Risk Factors

PCA reveals how many independent risk factors drive the asset universe.
If PC1 explains 40%+ of variance, the market is dominated by a single factor
(usually "risk-on / risk-off").

This has direct implications for portfolio construction:
- Fewer independent factors = less true diversification
- HRP and risk parity methods account for this

In [ ]:
pca_result = corr_analyzer.pca_analysis()

print(f"Components for 90% variance: {pca_result['n_components_90pct']}")
print(f"\nTop 5 components:")
for i in range(min(5, len(pca_result['explained_variance_ratio']))):
    print(f"  PC{i+1}: {pca_result['explained_variance_ratio'][i]:.1%} "
          f"(cumulative: {pca_result['cumulative_variance'][i]:.1%})")

In [ ]:
fig = viz.plot_pca(pca_result)
plt.show()

In [ ]:
# PC1 loadings — what does the dominant factor represent?
pc1 = pca_result['loadings']['PC1'].sort_values()

fig, ax = plt.subplots(figsize=(12, 8))
colors = [viz._get_color(t) for t in pc1.index]
pc1.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('PC1 Loadings — The Dominant Market Factor', fontsize=14, fontweight='bold')
ax.set_xlabel('Loading on PC1')
ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Correlation Clusters

Group assets by their correlation structure. Assets in the same cluster
move together — true diversification requires combining assets from
different clusters.

In [ ]:
clusters = corr_analyzer.correlation_clusters(n_clusters=5)
print('Correlation-Based Asset Clusters:')
print('=' * 60)
for c in sorted(clusters['Cluster'].unique()):
    members = clusters[clusters['Cluster'] == c]
    print(f"\nCluster {c}: {list(members.index)}")
    print(f"  Asset classes: {members['Asset_Class'].value_counts().to_dict()}")

## 12. Tail Dependence

Standard correlation measures linear co-movement.
**Tail dependence** measures the probability of simultaneous extreme events.

High tail dependence = assets crash together, even if average correlation is moderate.

In [ ]:
# Select a smaller subset for readability
td_tickers = ['XLK', 'XLF', 'EEM', 'BTC-USD', 'GLD', 'TLT', 'USO', 'VNQ']
td_tickers = [t for t in td_tickers if t in daily_returns.columns]

td_analyzer = CorrelationAnalyzer(daily_returns[td_tickers], asset_class_map=class_map)
tail_dep = td_analyzer.tail_dependence(threshold_pct=5.0)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(tail_dep, dtype=bool), k=1)
sns.heatmap(tail_dep, mask=mask, cmap='YlOrRd', annot=True, fmt='.2f',
            square=True, linewidths=0.5, ax=ax, vmin=0, vmax=0.5)
ax.set_title('Lower Tail Dependence (5th percentile)\nP(Y crashes | X crashes)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Key Takeaways from Module 2

1. **Returns are NOT normal** — all assets show fat tails and most exhibit excess kurtosis, confirming we need robust risk measures
2. **Student-t distribution** fits much better than Normal for most assets — we should use t-distribution for VaR/CVaR
3. **Volatility clusters** — GARCH persistence near 1.0 means volatility shocks last a long time; static volatility estimates are misleading
4. **Correlations increase in crises** — the diversification benefit partially evaporates when you need it most
5. **Few independent risk factors** — PCA shows the market is dominated by a small number of common factors
6. **Tail dependence is real** — some assets crash together more than linear correlation suggests

### Implications for Portfolio Optimization

- Use **CVaR** instead of VaR for risk measurement
- Use **shrinkage or robust covariance estimators** (Module 3) instead of sample covariance
- Consider **regime-aware allocation** that adapts to changing market conditions
- **HRP** is more robust than Markowitz because it doesn't require covariance matrix inversion

### Next: Module 3 — Covariance Estimation

We'll compare Sample, Ledoit-Wolf, Oracle Approximating Shrinkage, and DCC-GARCH covariance estimators.